# 01 - Build the coup event list

**Question:** do currency markets move more than normal around coups d'etat?

This notebook turns the raw Powell & Thyne coup data into a clean list of events (2000-2025) for the event study.

- **Source:** Powell & Thyne, Global Instances of Coups (version 2026-08-29)
- **Input:** `data/raw/pt_20260829.csv`
- **Output:** `data/clean/events_2000_2025.csv`
- **Coding:** in the source data, `coup` = 2 is a successful coup and `coup` = 1 is a failed attempt

In [ ]:
import pandas as pd

RAW_FILE = "../data/raw/pt_20260829.csv"
CLEAN_FILE = "../data/clean/events_2000_2025.csv"

START_YEAR = 2000
END_YEAR = 2025

## Load the raw data

In [ ]:
df = pd.read_csv(RAW_FILE)

print("rows, columns:", df.shape)
df.head()

## Add a proper date and a readable outcome label

In [ ]:
# Combine the year, month and day columns into one date.
# errors="coerce" turns any impossible date into a blank instead of crashing.
df["date"] = pd.to_datetime(df[["year", "month", "day"]], errors="coerce")

# Spell out the outcome so we don't have to remember what 1 and 2 mean.
df["outcome"] = df["coup"].map({2: "successful", 1: "failed"})

# Every row should have a valid date and a known outcome.
print("rows with an invalid date:", df["date"].isna().sum())
print("rows with an unknown outcome:", df["outcome"].isna().sum())

## Keep events from 2000 to 2025

The 2026 Niger attempt in the source file is excluded by this window (its outcome was still being coded).

In [ ]:
events = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)].copy()
events = events.sort_values("date").reset_index(drop=True)

print("events", START_YEAR, "-", END_YEAR, ":", len(events))
print()
print(events["outcome"].value_counts())
print()
print(events["country"].value_counts().head(15))

## Save the clean event list

In [ ]:
events.to_csv(CLEAN_FILE, index=False)
print("saved", len(events), "events to", CLEAN_FILE)